# 🛡️ DeepShield — Colab Training Notebook
> Dataset download → Google Drive mount → Training (Image / Audio / Video)

**Steps:**
1. Mount Google Drive
2. Set up Kaggle credentials
3. Download datasets
4. Organise folder structure
5. Clone / upload your backend code
6. Install requirements
7. Run training

## ✅ Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM (GB)  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## ✅ Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── All persistent data lives here on your Drive ──────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/DeepShield'
DATA_DIR     = f'{DRIVE_ROOT}/data'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'
KAGGLE_DIR   = f'{DRIVE_ROOT}/kaggle'

for d in [DATA_DIR, CKPT_DIR, RUNS_DIR, KAGGLE_DIR,
          f'{DATA_DIR}/images/real', f'{DATA_DIR}/images/fake',
          f'{DATA_DIR}/audio/real',  f'{DATA_DIR}/audio/fake',
          f'{DATA_DIR}/videos/real', f'{DATA_DIR}/videos/fake']:
    os.makedirs(d, exist_ok=True)

print('✅ Drive mounted. Root:', DRIVE_ROOT)

## ✅ Step 3 — Kaggle Credentials Setup

> **How to get your Kaggle API key:**
> 1. Go to https://www.kaggle.com/account
> 2. Scroll down → **Create New API Token**
> 3. It downloads `kaggle.json` — upload it when prompted below

In [ ]:
import os, json
from google.colab import files

KAGGLE_JSON_ON_DRIVE = f'{KAGGLE_DIR}/kaggle.json'

if os.path.exists(KAGGLE_JSON_ON_DRIVE):
    # Already saved on Drive from a previous session
    !mkdir -p ~/.kaggle
    !cp '{KAGGLE_JSON_ON_DRIVE}' ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    print('✅ kaggle.json loaded from Drive (no re-upload needed)')
else:
    print('Upload your kaggle.json file:')
    uploaded = files.upload()   # <-- pick kaggle.json from your computer
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    # Save to Drive for next time
    !cp kaggle.json '{KAGGLE_JSON_ON_DRIVE}'
    print('✅ kaggle.json saved to Drive for future sessions')

!pip install -q kaggle
!kaggle --version

## ✅ Step 4 — Download Datasets

Choose which datasets to download. The image datasets are the fastest to start with.

In [ ]:
# ── OPTION A: 140k Real vs Fake Faces (recommended — already structured) ──
# ~1.2 GB | Creative Commons licence
DOWNLOAD_140K   = True   # ← set True/False

# ── OPTION B: Deepfake and Real Images (Kaggle) ───────────────────────────
# ~700 MB | Creative Commons licence
DOWNLOAD_DFREAL = True   # ← set True/False

# ── OPTION C: WaveFake audio dataset ──────────────────────────────────────
# ~3 GB | MIT licence  (downloads from Zenodo, not Kaggle)
DOWNLOAD_WAVEFAKE = False  # ← set True/False

print('Selected downloads:')
print(f'  140k Faces   : {DOWNLOAD_140K}')
print(f'  DF Real      : {DOWNLOAD_DFREAL}')
print(f'  WaveFake     : {DOWNLOAD_WAVEFAKE}')

In [ ]:
import os, shutil, zipfile

def already_downloaded(marker_path):
    return os.path.exists(marker_path)

# ──────────────────────────────────────────────────────────────────────────
# OPTION A — 140k Real and Fake Faces
# ──────────────────────────────────────────────────────────────────────────
if DOWNLOAD_140K:
    MARKER_A = f'{DATA_DIR}/images/.140k_done'
    if already_downloaded(MARKER_A):
        print('✅ 140k dataset already on Drive, skipping download.')
    else:
        print('⬇️  Downloading 140k Real and Fake Faces...')
        !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /tmp/140k
        print('📦 Extracting...')
        with zipfile.ZipFile('/tmp/140k/140k-real-and-fake-faces.zip', 'r') as z:
            z.extractall('/tmp/140k_extracted')

        # The zip contains: real_vs_fake/train/real and real_vs_fake/train/fake
        # Copy into our Drive structure
        import glob
        for label in ['real', 'fake']:
            src_dir = f'/tmp/140k_extracted/real_vs_fake/train/{label}'
            if not os.path.isdir(src_dir):
                # Try alternate path in zip
                matches = glob.glob(f'/tmp/140k_extracted/**/{label}', recursive=True)
                if matches:
                    src_dir = matches[0]
            dst_dir = f'{DATA_DIR}/images/{label}'
            print(f'  Copying {label}/ → Drive...')
            for f in os.listdir(src_dir):
                shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))

        open(MARKER_A, 'w').close()  # mark as done
        print(f'✅ 140k dataset copied to {DATA_DIR}/images/')

# ──────────────────────────────────────────────────────────────────────────
# OPTION B — Deepfake and Real Images
# ──────────────────────────────────────────────────────────────────────────
if DOWNLOAD_DFREAL:
    MARKER_B = f'{DATA_DIR}/images/.dfreal_done'
    if already_downloaded(MARKER_B):
        print('✅ DF-Real dataset already on Drive, skipping download.')
    else:
        print('⬇️  Downloading Deepfake and Real Images...')
        !kaggle datasets download -d manjilkarki/deepfake-and-real-images -p /tmp/dfreal
        print('📦 Extracting...')
        with zipfile.ZipFile('/tmp/dfreal/deepfake-and-real-images.zip', 'r') as z:
            z.extractall('/tmp/dfreal_extracted')

        # Expected structure: Dataset/Train/Real and Dataset/Train/Fake
        for label, folder in [('real', 'Real'), ('fake', 'Fake')]:
            src_dir = f'/tmp/dfreal_extracted/Dataset/Train/{folder}'
            dst_dir = f'{DATA_DIR}/images/{label}'
            if os.path.isdir(src_dir):
                print(f'  Copying {folder}/ → Drive...')
                for f in os.listdir(src_dir):
                    shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))
            else:
                print(f'  ⚠️  Could not find {src_dir} — check zip structure.')
                !find /tmp/dfreal_extracted -type d | head -20

        open(MARKER_B, 'w').close()
        print(f'✅ DF-Real dataset copied to {DATA_DIR}/images/')

# ──────────────────────────────────────────────────────────────────────────
# OPTION C — WaveFake (audio)
# ──────────────────────────────────────────────────────────────────────────
if DOWNLOAD_WAVEFAKE:
    MARKER_C = f'{DATA_DIR}/audio/.wavefake_done'
    if already_downloaded(MARKER_C):
        print('✅ WaveFake already on Drive, skipping download.')
    else:
        print('⬇️  Downloading WaveFake from Zenodo (~3 GB)...')
        !pip install -q zenodo_get
        !zenodo_get 5642694 -o /tmp/wavefake
        print('📦 Extracting WaveFake...')
        !find /tmp/wavefake -name '*.gz' | xargs -I{} tar -xzf {} -C /tmp/wavefake_extracted --one-top-level=wavefake
        # ljspeech_* folders → real; all other vocoder folders → fake
        import glob
        for wav in glob.glob('/tmp/wavefake/**/*.wav', recursive=True):
            folder = wav.split('/')[-2]
            dst = f'{DATA_DIR}/audio/real' if 'ljspeech' in folder.lower() else f'{DATA_DIR}/audio/fake'
            shutil.copy(wav, os.path.join(dst, os.path.basename(wav)))
        open(MARKER_C, 'w').close()
        print(f'✅ WaveFake copied to {DATA_DIR}/audio/')

In [ ]:
# ── Verify dataset counts ─────────────────────────────────────────────────
import os
print('\n📊 Dataset Summary:')
print('─' * 40)
for modality in ['images', 'audio', 'videos']:
    for label in ['real', 'fake']:
        path = f'{DATA_DIR}/{modality}/{label}'
        if os.path.isdir(path):
            n = len([f for f in os.listdir(path) if not f.startswith('.')])
            status = '✅' if n > 0 else '⚠️ '
            print(f'  {status} {modality}/{label}: {n:,} files')
        else:
            print(f'  ⬜ {modality}/{label}: (not downloaded)')

## ✅ Step 5 — Clone / Upload DeepShield Backend

In [ ]:
# ── Option A: Clone from GitHub (if your backend is public or private with token) ──
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/deepfake.git'   # ← change this
BACKEND_DIR = '/content/deepshield_backend'

if not os.path.isdir(BACKEND_DIR):
    !git clone {GITHUB_REPO} {BACKEND_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {BACKEND_DIR} pull

print('✅ Backend ready at:', BACKEND_DIR)

In [ ]:
# ── Option B: Upload a zip of the backend manually ──────────────────────
# Run this cell INSTEAD of the one above if you don't have a GitHub repo

# from google.colab import files
# uploaded = files.upload()   # upload backend.zip
# !unzip backend.zip -d /content/deepshield_backend
# BACKEND_DIR = '/content/deepshield_backend/backend'
# print('✅ Backend extracted to:', BACKEND_DIR)

## ✅ Step 6 — Install Requirements

In [ ]:
BACKEND_DIR = '/content/deepshield_backend/backend'   # adjust if needed

!pip install -q -r {BACKEND_DIR}/requirements.txt
print('✅ Requirements installed')

## ✅ Step 7A — Train Image Model 🖼️

In [ ]:
import os, sys
BACKEND_DIR = '/content/deepshield_backend/backend'
sys.path.insert(0, BACKEND_DIR)
os.chdir(BACKEND_DIR)

# ── Hyperparameters — tweak as needed ────────────────────────────────────
EPOCHS      = 40
BATCH_SIZE  = 32
LR          = 1e-4
IMAGE_SIZE  = 224
WORKERS     = 2

# ── Paths ─────────────────────────────────────────────────────────────────
IMAGE_DATA  = f'{DATA_DIR}/images'       # must have real/ and fake/ inside
CKPT_PATH   = f'{CKPT_DIR}/image'
TB_PATH     = f'{RUNS_DIR}/image'

!python training/image_train.py \
    --data_dir        {IMAGE_DATA} \
    --epochs          {EPOCHS} \
    --batch_size      {BATCH_SIZE} \
    --lr              {LR} \
    --image_size      {IMAGE_SIZE} \
    --num_workers     {WORKERS} \
    --checkpoint_dir  {CKPT_PATH} \
    --tensorboard_dir {TB_PATH}

## ✅ Step 7B — Train Audio Model 🔊

In [ ]:
AUDIO_DATA  = f'{DATA_DIR}/audio'        # must have real/ and fake/ inside
CKPT_PATH_A = f'{CKPT_DIR}/audio'
TB_PATH_A   = f'{RUNS_DIR}/audio'

!python training/audio_train.py \
    --data_dir        {AUDIO_DATA} \
    --epochs          30 \
    --batch_size      32 \
    --checkpoint_dir  {CKPT_PATH_A} \
    --tensorboard_dir {TB_PATH_A}

## ✅ Step 7C — Train All Models (Image + Audio + Video) 🚀

In [ ]:
# Runs image, audio, and video training sequentially
!python training/train_all.py \
    --image_dir       {DATA_DIR}/images \
    --audio_dir       {DATA_DIR}/audio \
    --video_dir       {DATA_DIR}/videos \
    --checkpoint_dir  {CKPT_DIR} \
    --tensorboard_dir {RUNS_DIR}

## ✅ Step 8 — View TensorBoard Logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR}

## ✅ Step 9 — Copy Trained Checkpoints back to Drive
> Checkpoints are already saved directly to Drive (`CKPT_DIR`), so nothing extra to do! They persist across sessions automatically.

In [ ]:
# Verify checkpoint files saved on Drive
!find {CKPT_DIR} -name '*.pt' -o -name '*.pth' 2>/dev/null | sort
print('\nAll checkpoints live on Google Drive → safe even after Colab disconnects.')